# Setup

In [85]:
import io
import os       
import requests
import pandas as pd
import duckdb
import yfinance as yf
from tqdm.auto import tqdm
from dotenv import load_dotenv
from fredapi import Fred

# Data Ingestion

In [9]:
PATH_MAIN = os.path.join(os.getcwd(), 'data')
PATH_HOLDINGS = f"{PATH_MAIN}/holdings"
PATH_PRICES = f"{PATH_MAIN}/prices"
PATH_CALENDARS = f"{PATH_MAIN}/calendars"

os.makedirs(PATH_MAIN, exist_ok=True)
os.makedirs(PATH_HOLDINGS, exist_ok=True)
os.makedirs(PATH_PRICES, exist_ok=True)
os.makedirs(PATH_CALENDARS, exist_ok=True)

ETFS = [
  'XLC', 'XLY', 'XLP', 'XLE', 'XLF', 'XLV', 'XLI', 'XLB', 'XLRE', 'XAR',
  'KBE', 'XBI', 'KCE', 'XHE', 'XHS', 'XHB', 'KIE', 'XME', 'XES', 'XOP',
  'XPH', 'KRE', 'XRT', 'XSD', 'XSW', 'XTL', 'XTN', 'XLK', 'XLU'
]

ETFS_TEST = ['XLE', 'XLU']

In [51]:
# download official spdr holdings

def download_holdings(tickers, path=PATH_HOLDINGS):
    for etf in tqdm(tickers):
        try:
            ssga_url = f'https://www.ssga.com/us/en/intermediary/library-content/products/fund-data/etfs/us/holdings-daily-us-en-{etf.lower()}.xlsx'
            headers = {"User-Agent": "Mozilla/5.0"}
            response = requests.get(ssga_url, headers=headers)
            response.raise_for_status()
            # save and partially clean
            duckdb.sql(f"""
                copy (
                    select
                        '{etf}' as etf
                        , Name as name
                        , Ticker as ticker
                        , Weight::numeric as weight
                    from read_xlsx('{ssga_url}', range = 'A5:H')
                    where true
                        and SEDOL != '-'
                        and ticker != '-'
                ) to '{path}/{etf.lower()}_holdings.parquet'
            """)
        except Exception as e:
            print(f"{etf} error: {e}")
            continue
    # save consolidated holdings
    duckdb.sql(f"""
        copy (
            select *
            from read_parquet("{path}/*.parquet", union_by_name=True)
        ) to "{PATH_MAIN}/holdings.parquet"
    """)

In [20]:
download_holdings(tickers=ETFS_TEST)

100%|██████████| 2/2 [00:03<00:00,  1.82s/it]


In [22]:
holdings = duckdb.sql(f"""
    select *
    from read_parquet("{PATH_MAIN}/holdings.parquet")
""").fetchdf()

In [23]:
print(holdings.info())

<class 'pandas.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   etf     52 non-null     str    
 1   name    52 non-null     str    
 2   ticker  52 non-null     str    
 3   weight  52 non-null     float64
dtypes: float64(1), str(3)
memory usage: 2.9 KB
None


In [24]:
print(holdings.head().sort_values("weight", ascending=False))

   etf                      name ticker  weight
0  XLE  EXXONMOBIL HOLDINGS CORP    XOM  20.099
1  XLE              CHEVRON CORP    CVX  14.834
2  XLE            CONOCOPHILLIPS    COP   5.979
3  XLE   MARATHON PETROLEUM CORP    MPC   4.858
4  XLE               PHILLIPS 66    PSX   4.832


In [53]:
# download prices from yfinance

def download_prices(tickers, path=PATH_PRICES, interval="1d", period="5y"):
    unique_tickers = tickers.ticker.unique()
    print("Removed duplicate names.")
    unique_tickers = [ticker.replace('.', '-') for ticker in unique_tickers]
    print("Fixed ticker names.")
    for ticker in tqdm(unique_tickers):
        try:
            df = yf.download(
                ticker
                , interval=interval
                , period=period
                , multi_level_index=False
                , progress=False
                , auto_adjust=True
            ).reset_index()
            # save and partially clean
            duckdb.sql(f"""
                copy (
                    select
                        '{ticker}' as ticker
                        , Date::date as date
                        , Open as open
                        , High as high
                        , Low as low
                        , Close as close
                        , Volume as volume
                        , (close*volume)::bigint as value
                    from df
                ) to '{path}/{ticker.lower()}_prices.parquet'
            """)
        except Exception as e:
            print(f"{ticker} error: {e}")
            continue
    # save consolidated prices
    duckdb.sql(f"""
        copy (
            select *
            from read_parquet("{path}/*.parquet", union_by_name=True)
        ) to "{PATH_MAIN}/prices.parquet"
    """)    

In [28]:
download_prices(holdings)

Removed duplicate names.
Fixed ticker names.


100%|██████████| 52/52 [00:46<00:00,  1.11it/s]


In [55]:
prices = duckdb.sql(f"""
    select *
    from read_parquet("{PATH_MAIN}/prices.parquet")
""").fetchdf()

In [56]:
print(prices.info())

<class 'pandas.DataFrame'>
RangeIndex: 65078 entries, 0 to 65077
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   ticker  65078 non-null  str           
 1   date    65078 non-null  datetime64[us]
 2   open    65078 non-null  float64       
 3   high    65078 non-null  float64       
 4   low     65078 non-null  float64       
 5   close   65078 non-null  float64       
 6   volume  65078 non-null  int64         
 7   value   65078 non-null  int64         
dtypes: datetime64[us](1), float64(4), int64(2), str(1)
memory usage: 4.2 MB
None


In [57]:
print(prices.head())

  ticker       date       open       high        low      close   volume  \
0    AEE 2021-07-15  71.700537  72.780104  71.614173  72.607376  1462400   
1    AEE 2021-07-16  72.728290  73.807858  72.460560  73.462395   808600   
2    AEE 2021-07-19  73.237821  73.876922  71.657333  72.529625  1471100   
3    AEE 2021-07-20  72.624621  73.505551  72.313712  72.529625   949100   
4    AEE 2021-07-21  72.495089  72.529636  71.052782  71.078697  1210400   

       value  
0  106181027  
1   59401692  
2  106698331  
3   68837867  
4   86033655  


In [65]:
# download calendars from yfinance

def download_calendars(tickers, path=PATH_CALENDARS):
    unique_tickers = tickers.ticker.unique()
    print("Removed duplicate names.")
    unique_tickers = [ticker.replace('.', '-') for ticker in unique_tickers]
    print("Fixed ticker names.")
    for ticker in tqdm(unique_tickers):
        try:
            df = yf.Ticker(ticker).earnings_dates.reset_index()
            # save and partially clean
            duckdb.sql(f"""
                copy (
                    select
                        '{ticker}' as ticker
                        , "Earnings Date"::date as earnings_date
                        , "EPS Estimate" as eps_estimate
                        , "Reported EPS" as eps_actual
                        , "Surprise(%)" as surprise
                    from df
                ) to '{path}/{ticker.lower()}_calendars.parquet'
            """)
        except Exception as e:
            print(f"{ticker} error: {e}")
            continue
    # save consolidated calendars
    duckdb.sql(f"""
        copy (
            select *
            from read_parquet("{path}/*.parquet", union_by_name=True)
        ) to "{PATH_MAIN}/calendars.parquet"
    """)    

In [66]:
download_calendars(holdings)

Removed duplicate names.
Fixed ticker names.


 33%|███▎      | 17/52 [00:02<00:04,  7.94it/s]

EQT error: ['Earnings Date']


 90%|█████████ | 47/52 [00:05<00:00,  8.34it/s]

CMS error: ['Earnings Date']


100%|██████████| 52/52 [00:06<00:00,  8.41it/s]


In [74]:
calendars = duckdb.sql(f"""
    select ticker
        , earnings_date
    from read_parquet("{PATH_MAIN}/calendars.parquet")
""").fetchdf()

In [76]:
print(calendars.info())

<class 'pandas.DataFrame'>
RangeIndex: 1241 entries, 0 to 1240
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   ticker         1241 non-null   str           
 1   earnings_date  1241 non-null   datetime64[us]
dtypes: datetime64[us](1), str(1)
memory usage: 23.1 KB
None


In [77]:
print(calendars.head())

  ticker earnings_date
0    AEE    2026-07-31
1    AEE    2026-05-06
2    AEE    2026-02-12
3    AEE    2025-11-06
4    AEE    2025-08-01


In [94]:
# download sentiment proxies from fred

load_dotenv()
fred_api_key = os.getenv('fred')
fred = Fred(api_key=fred_api_key)

# CBOE Volatility Index: VIX (VIXCLS)
vix = fred.get_series('VIXCLS')
vix.name = 'vix'
# ICE BofA AAA US Corporate Index Option-Adjusted Spread (BAMLC0A1CAAA)
aaa = fred.get_series('BAMLC0A1CAAA')
aaa.name = 'aaa'
# ICE BofA BBB US Corporate Index Option-Adjusted Spread (BAMLC0A4CBBB)
bbb = fred.get_series('BAMLC0A4CBBB')
bbb.name = 'bbb'

# Combine all three to a single dataframe, matched by date
sentiment_proxies = pd.concat([vix, aaa, bbb], axis=1, sort=False).reset_index().to_parquet(f"{PATH_MAIN}/sentiment.parquet")

In [102]:
sentiment = duckdb.sql(f"""
    select index::date as date
        , vix
        , aaa
        , bbb
    from read_parquet("{PATH_MAIN}/sentiment.parquet")
""").fetchdf()

In [104]:
print(sentiment.info())

<class 'pandas.DataFrame'>
RangeIndex: 9542 entries, 0 to 9541
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    9542 non-null   datetime64[us]
 1   vix     9228 non-null   float64       
 2   aaa     785 non-null    float64       
 3   bbb     785 non-null    float64       
dtypes: datetime64[us](1), float64(3)
memory usage: 298.3 KB
None


In [105]:
print(sentiment.head())

        date    vix  aaa  bbb
0 1990-01-02  17.24  NaN  NaN
1 1990-01-03  18.19  NaN  NaN
2 1990-01-04  19.22  NaN  NaN
3 1990-01-05  20.11  NaN  NaN
4 1990-01-08  20.26  NaN  NaN


# Data Cleaning

# Data Processing

# Modeling

# Conclusion